# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question: Are ratings different across genres, languages, or release periods?
- Why this question matters: Knowing this information can help distinguish what makes certain movies have higher ratings overall compared to other ones, and possibly help establish how movies coming out in the future will rate.
- Tables/files used: movies, ratings, links, credits, keywords.
- Row meaning of each important table:
Movie: each row means a different movie
Rating: each row means a rating that an individual gave for a specific movie
Links: each row gives the IMBD ID and TMBD ID for a specific movie
Credits: each rows gives the director and top cast for a specifc TMBD ID
Keywords: each row gives keywords used to describe a specifc movie based on its TMBD ID
- Early data quality issues: there are missing values for tables movies, links, credits, and keywords.
- Initial cleaning plan: handle the missing values such as the ones in primary genre, and replace the hidden missing values in buget, revenue. Check duplicate keys. Handle sparce data such as the movies with ratings that are fewer than 10 or have only 1. 
- Possible target/outcome: have combined categories aggregated to have an overall understanding of which factors differentiate ratings.
- Ethics, privacy, or governance concerns: the majority of movies in the dataset are english language, which could make non-english films underrepresented in the set. 
--------------------------------------------------------------------------
##Copied questions from Canvas
- Business problem draft: A streaming or content platform wants to know what makes a movie well rated by factors such as genre, language, or release period. 
- Intended user or decision context: the intended user would be production teams or movie production companies that want to know what makes movies rate higher.
- Dataset description and source link: This project uses a curated version of The Movies Dataset, which combines movie metadata from The Movie Database (TMDB) with user ratings from MovieLens, collected by the GroupLens research lab.
- List of raw files or tables: movies, ratings, links, credits, keywords
- Explanation of what one row means in each important table:Movie: each row means a different movie
Rating: each row means a rating that an individual gave for a specific movie
Links: each row gives the IMBD ID and TMBD ID for a specific movie
Credits: each rows gives the director and top cast for a specifc TMBD ID
Keywords: each row gives keywords used to describe a specifc movie based on its TMBD ID
- Initial row and column counts: movies:   (9082, 21)
ratings:  (99810, 5)
links:    (9125, 3)
credits:  (9082, 3)
keywords: (9082, 2)
- Data types: int64, str, bool, float64
- Missing values: there are missing values for tables movies, links, credits, and keywords.
- Duplicate records or duplicate keys: none
- Suspicious, impossible, or confusing values: we have values that are blank or have a "0" in budget and revenue, which is a hidden blank that we have to fix. 
- Initial cleaning plan: handle the missing values such as the ones in primary genre, and replace the hidden missing values in buget, revenue. Check duplicate keys. Handle sparce data such as the movies with ratings that are fewer than 10 or have only 1. 
- Potential target or outcome variable: possibly create average rating per movie. 
- Early ethics, privacy, governance, or responsible-use concerns:  the majority of movies in the dataset are english language, which could make non-english films underrepresented in the set. 



In [6]:
## Step 1, print out the shape to understand our data shape for each table
print("Rows and columns in each table:")
print("movies:  ", movies.shape)
print("ratings: ", ratings.shape)
print("links:   ", links.shape)
print("credits: ", credits.shape)
print("keywords:", keywords.shape)
 

Rows and columns in each table:
movies:   (9082, 21)
ratings:  (99810, 5)
links:    (9125, 3)
credits:  (9082, 3)
keywords: (9082, 2)


Step 2: Pick the columns that matter for our research question

In [7]:
movie_cols = ["movieId", "title", "primary_genre", "genres_list",
              "original_language", "release_date", "release_year",
              "runtime", "budget", "revenue", "vote_average", "vote_count"]
rating_cols = ["userId", "movieId", "rating", "timestamp"]

Step 3: Find blank and missing values as well as percent of each table

In [8]:
movie_missing = pd.DataFrame({
    "missing_count": movies[movie_cols].isna().sum(),
})
print("Movies - blank values:")
print(movie_missing)

Movies - blank values:
                   missing_count
movieId                        0
title                          0
primary_genre                 35
genres_list                   35
original_language              0
release_date                   0
release_year                   0
runtime                        0
budget                         0
revenue                        0
vote_average                   0
vote_count                     0


In [9]:
print("Hidden missing values in movies (0 or '[]'):")
print("budget = 0:", (movies["budget"] == 0).sum())
print("revenue = 0:", (movies["revenue"] == 0).sum())
print("runtime = 0:", (movies["runtime"] == 0).sum())
print("vote_count = 0:", (movies["vote_count"] == 0).sum())
print("genres = '[]':", (movies["genres"] == "[]").sum())
 

Hidden missing values in movies (0 or '[]'):
budget = 0: 4443
revenue = 0: 4322
runtime = 0: 20
vote_count = 0: 44
genres = '[]': 35


We can see that there are many missing values that are either 0 or blank which could indicate unknown values. 

PART 2: Duplicate keys check 


In [10]:
print("Exact duplicate rows:")
print("movies:  ", movies.duplicated().sum())
print("ratings: ", ratings.duplicated().sum())
print("links:   ", links.duplicated().sum())
print("credits: ", credits.duplicated().sum())
print("keywords:", keywords.duplicated().sum())

Exact duplicate rows:
movies:   0
ratings:  0
links:    0
credits:  0
keywords: 0


In [11]:
print("Duplicate key values:")
print("movies  movieId:", movies["movieId"].duplicated().sum())
print("movies  tmdbId:", movies["tmdbId"].duplicated().sum())
print("movies  imdb_id:", movies["imdb_id"].duplicated().sum())
print("links   movieId:", links["movieId"].duplicated().sum())
print("credits tmdbId:", credits["tmdbId"].duplicated().sum())
print("keywords tmdbId:", keywords["tmdbId"].duplicated().sum())

Duplicate key values:
movies  movieId: 0
movies  tmdbId: 0
movies  imdb_id: 0
links   movieId: 0
credits tmdbId: 0
keywords tmdbId: 0


We can see there are no duplicate keys and no duplicate rows for our datasets.

Part 3: Suspicious value checks 

We check to see if there are any outliers in the ratings.

In [12]:
bad_rating = ratings[(ratings["rating"] < 0.5) | (ratings["rating"] > 5)]
print("Ratings outside 0.5-5:      ", len(bad_rating))
print(ratings["rating"].value_counts().sort_index())

Ratings outside 0.5-5:       0
rating
0.5     1097
1.0     3322
1.5     1685
2.0     7263
2.5     4446
3.0    20039
3.5    10512
4.0    28705
4.5     7698
5.0    15043
Name: count, dtype: int64


We see there are none in our ratings table.

We then want to check if there are any typos in budget / revenue

In [13]:
tiny_money = movies[((movies["budget"] > 0) & (movies["budget"] < 1000)) |
                    ((movies["revenue"] > 0) & (movies["revenue"] < 1000))]
print("Budget or revenue under $1,000:", len(tiny_money))
print(tiny_money[["title", "budget", "revenue"]].head(10))

Budget or revenue under $1,000: 70
                     title  budget  revenue
82             Black Sheep       0     32.0
105  Anne Frank Remembered       0      1.0
161           Mute Witness       2      1.0
166           The Prophecy       8     16.0
221       Immortal Beloved     120      0.0
250         Miami Rhapsody       6      5.0
289          The Sum of Us      32      0.0
408            The Getaway       0     30.0
488      Striking Distance      30      0.0
610             Last Dance       0      5.0


We see that there is multiple movies that have low budget and revenue, these look suspicious in our data and can potentially not be included in our later analysis.

We then check the release date by year range for the movies in the dataset. 

In [14]:

movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
print("Release year range:", movies["release_year"].min(), "to", movies["release_year"].max())
print("Bad / unreadable release dates:", movies["release_date"].isna().sum())

Release year range: 1902 to 2016
Bad / unreadable release dates: 0


We are now going to look at movie status values to check for any suspicious values in our sets. Additionally, we are going to look at the top 15 languages to see if there are any discrepencies or disproportionate values. Lastly, we are going to look at primary genre by totals to see the different amounts of each primary genre for our analysis.

In [15]:
print("Movie status values:")
print(movies["status"].value_counts(dropna=False))
 
print("Top 15 languages:")
print(movies["original_language"].value_counts().head(15))
 
print("Primary genre counts:")
print(movies["primary_genre"].value_counts(dropna=False))

Movie status values:
status
Released           9061
Rumored              11
Post Production       7
NaN                   2
In Production         1
Name: count, dtype: int64
Top 15 languages:
original_language
en    7947
fr     272
ja     185
de     113
it      98
es      88
cn      47
sv      41
zh      39
ko      31
da      30
ru      27
pt      25
nl      16
fa      14
Name: count, dtype: int64
Primary genre counts:
primary_genre
Drama              2413
Comedy             2168
Action             1177
Adventure           517
Horror              497
Crime               417
Documentary         383
Thriller            263
Fantasy             244
Animation           236
Romance             189
Science Fiction     126
Mystery              96
Music                82
Family               76
War                  52
Western              48
History              44
NaN                  35
TV Movie             18
Foreign               1
Name: count, dtype: int64


We can see that there is a disproportionate amount of movies in English, which could provide an uneven amount of sample data for the other languages when analyzing our data. Additionally, we see that the top 3 genres that hold a high percentage of the movies is Drama, Comedy, and Action. 

Now, we wil check for movies with vary few ratings, this will help us determine the validity of the ratings based on the sample sizes of some movies.

In [16]:
ratings_per_movie = ratings.groupby("movieId").size()
print("Movies with only 1 rating:  ", (ratings_per_movie == 1).sum())
print("Movies with fewer than 10:  ", (ratings_per_movie < 10).sum())
 

Movies with only 1 rating:   3042
Movies with fewer than 10:   6784


We see that there are over 3,000 movies with only 1 rating, which will not be a reliable rating compared to movies with much more data. Additionally, we have over 6000 movies with fewer than 10 ratings for them, which is not as reliable either. 

In [17]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,datetime64[us]
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [18]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.
total_rated_movies = ratings_per_movie.shape[0]
print("Total movies with at least one rating:", total_rated_movies)

print("Movies with 1+ ratings: ", (ratings_per_movie >= 1).sum())
print("Movies with 5+ ratings: ", (ratings_per_movie >= 5).sum())
print("Movies with 10+ ratings:", (ratings_per_movie >= 10).sum())
print("Movies with 20+ ratings:", (ratings_per_movie >= 20).sum())
print("Movies with 50+ ratings:", (ratings_per_movie >= 50).sum())

movies_20plus_ids = ratings_per_movie[ratings_per_movie >= 20].index
eligible_movies = movies[movies["movieId"].isin(movies_20plus_ids)].copy()

print("Movies kept for the analysis:", eligible_movies.shape[0])

eligible_movies["release_decade"] = (eligible_movies["release_year"] // 10) * 10

print()
print("Movies with 20+ ratings by primary genre:")
print(eligible_movies["primary_genre"].value_counts())

print()
print("Movies with 20+ ratings by language (top 8):")
print(eligible_movies["original_language"].value_counts().head(8))

print()
print("Movies with 20+ ratings by release decade:")
print(eligible_movies["release_decade"].value_counts().sort_index())

Total movies with at least one rating: 9025
Movies with 1+ ratings:  9025
Movies with 5+ ratings:  3485
Movies with 10+ ratings: 2241
Movies with 20+ ratings: 1300
Movies with 50+ ratings: 453
Movies kept for the analysis: 1300

Movies with 20+ ratings by primary genre:
primary_genre
Comedy             301
Drama              293
Action             182
Adventure          150
Crime               82
Fantasy             55
Horror              46
Science Fiction     39
Animation           38
Thriller            34
Romance             22
Mystery             16
Family              12
Music                8
History              6
Documentary          5
Western              5
War                  5
Name: count, dtype: int64

Movies with 20+ ratings by language (top 8):
original_language
en    1253
ja      11
fr       9
de       8
it       6
es       6
zh       3
cn       1
Name: count, dtype: int64

Movies with 20+ ratings by release decade:
release_decade
1920      2
1930     12
1940     22
19

Here I checked how many ratings each movie has and how many are at each rating level. We then keep only movies with 20 or more ratings and create a release decade label for each movie.  

In [19]:
movies_clean = movies.copy()

print("BEFORE cleaning:")
print("Budget under $1,000:  ", (movies_clean["budget"] < 1000).sum())
print("Revenue under $1,000: ", (movies_clean["revenue"] < 1000).sum())
print("Runtime of 0:         ", (movies_clean["runtime"] == 0).sum())
print("Vote count of 0:      ", (movies_clean["vote_count"] == 0).sum())
print("Missing primary genre:", movies_clean["primary_genre"].isna().sum())

movies_clean["budget"] = movies_clean["budget"].where(movies_clean["budget"] >= 1000)
movies_clean["revenue"] = movies_clean["revenue"].where(movies_clean["revenue"] >= 1000)

movies_clean["runtime"] = movies_clean["runtime"].where(movies_clean["runtime"] > 0)
movies_clean["runtime_suspicious"] = movies_clean["runtime"] > 300

has_votes = movies_clean["vote_count"] > 0
movies_clean["vote_average"] = movies_clean["vote_average"].where(has_votes)
movies_clean["vote_count"] = movies_clean["vote_count"].where(has_votes)

movies_clean["primary_genre"] = movies_clean["primary_genre"].fillna("Unknown")
movies_clean["genres_list"] = movies_clean["genres_list"].fillna("Unknown")

movies_clean["is_released"] = movies_clean["status"] == "Released"

movies_clean = movies_clean.drop(columns= ["genres", "production_companies", "production_countries", "adult", "original_title"])

print()
print("AFTER cleaning (now counted as missing):")
print("Missing budget:      ", movies_clean["budget"].isna().sum())
print("Missing revenue:     ", movies_clean["revenue"].isna().sum())
print("Missing runtime:     ", movies_clean["runtime"].isna().sum())
print("Missing vote average:", movies_clean["vote_average"].isna().sum())
print("Unknown genre:       ", (movies_clean["primary_genre"] == "Unknown").sum())
print("Not released:        ", (movies_clean["is_released"] == False).sum())
print("Very long runtime:   ", movies_clean["runtime_suspicious"].sum())

print()
print("Rows before cleaning:", movies.shape[0])
print("Rows after cleaning: ", movies_clean.shape[0])

BEFORE cleaning:
Budget under $1,000:   4483
Revenue under $1,000:  4369
Runtime of 0:          20
Vote count of 0:       44
Missing primary genre: 35

AFTER cleaning (now counted as missing):
Missing budget:       4483
Missing revenue:      4369
Missing runtime:      20
Missing vote average: 44
Unknown genre:        35
Not released:         21
Very long runtime:    15

Rows before cleaning: 9082
Rows after cleaning:  9082


Here I made a copty of the original table 'movies' and called it 'movies_clean' so the raw data stays unchanged to compare the before and after. I turned hidden missing values into true missing values such as for budget and revenue where any value under $1,000 was changed to missing, including the onces with a value of 0. The movies with a runtime of 0 were also changed to missing. Movies with a vote average and vote count of 0 were changed to missing. Movies with no primary genre were labeled as unknown genre. Also sorted the movies as a 'is_released' to filter out movies that were rumored, in post production, in production or have no status. I dropped 'genres', 'production_countries', and 'production_companies' because they are not needed for our analysis and 'primary_genre' captures the necessary genre information for our analysis.  

In [20]:

movies_clean["status"] = movies_clean["status"].fillna("Unknown")

print("Data types after cleaning:")
print(movies_clean[["release_date", "release_year", "budget", "revenue", "runtime",
                    "vote_average", "vote_count", "title", "original_language",
                    "primary_genre", "status"]].dtypes)

movies_clean["title"] = movies_clean["title"].str.strip()
movies_clean["primary_genre"] = movies_clean["primary_genre"].str.strip()
movies_clean["original_language"] = movies_clean["original_language"].str.strip().str.lower()

top_languages = movies_clean["original_language"].value_counts().head(7).index

def group_language(language):
    if language in top_languages:
        return language
    else:
        return "other"

movies_clean["language_group"] = movies_clean["original_language"].apply(group_language)
movies_clean["is_english"] = movies_clean["original_language"] == "en"
movies_clean["release_decade"] = (movies_clean["release_year"] // 10) * 10
movies_clean["has_budget"] = movies_clean["budget"].notna()

print()
print("Movies by language group:")
print(movies_clean["language_group"].value_counts())

print()
print("Movies by release decade:")
print(movies_clean["release_decade"].value_counts().sort_index())

print()
print("Movies with a valid budget:", movies_clean["has_budget"].sum())

print()
print("Rows after transformations:", movies_clean.shape[0])


Data types after cleaning:
release_date         datetime64[us]
release_year                  int64
budget                      float64
revenue                     float64
runtime                     float64
vote_average                float64
vote_count                  float64
title                           str
original_language               str
primary_genre                   str
status                          str
dtype: object

Movies by language group:
language_group
en       7947
other     332
fr        272
ja        185
de        113
it         98
es         88
cn         47
Name: count, dtype: int64

Movies by release decade:
release_decade
1900       1
1910       6
1920      56
1930     148
1940     250
1950     341
1960     443
1970     549
1980    1193
1990    2193
2000    2533
2010    1369
Name: count, dtype: int64

Movies with a valid budget: 4599

Rows after transformations: 9082


Here I filled the 2 missing status values with "Unknown." I removed extra spaces from 'title', 'primary_genre', and 'original_language', and made language codes lowercase so the same language is always written the same way.'language_group' keeps the seven most common languages and labels the rest "other," since many languages have too few movies to compare on their own. 'is_english marks English versus non-English movie. 'release_decade' groups each movie by decade to answer the release-period part of my question. 'has_budget' marks the movies that still have a valid budget after cleaning. The table still has 9,082 rows, so these steps added columns without adding or removing any movies.

In [21]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies_clean
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [22]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies_clean),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9082
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    1300
dtype: int64

,movieId,tmdbId,imdb_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,genres_list,primary_genre,release_year,runtime_suspicious,is_released,language_group,is_english,release_decade,has_budget,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,21.946943,7.7,5415.0,Released,Animation|Comedy|Family,Animation,1995,False,True,en,True,1990,True,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,17.015539,6.9,2413.0,Released,Adventure|Fantasy|Family,Adventure,1995,False,True,en,True,1990,True,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,en,1995-12-22,101.0,NaN,NaN,11.712900,6.5,92.0,Released,Romance|Comedy,Romance,1995,False,True,en,True,1990,False,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,3.859495,6.1,34.0,Released,Comedy|Drama|Romance,Comedy,1995,False,True,en,True,1990,True,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,en,1995-02-10,106.0,NaN,76578911.0,8.387519,5.7,173.0,Released,Comedy,Comedy,1995,False,True,en,True,1990,False,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
budget,0.493614
revenue,0.481061
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
user_rating_mean,0.006276
first_rating_date,0.006276
user_rating_count,0.006276


In [23]:
print("movieId type in movies_clean:  ", movies_clean["movieId"].dtype)
print("movieId type in rating_summary:", rating_summary["movieId"].dtype)
print("tmdbId type in movies_clean:   ", movies_clean["tmdbId"].dtype)
print("tmdbId type in credits:        ", credits["tmdbId"].dtype)
print("tmdbId type in keywords:       ", keywords["tmdbId"].dtype)

ratings_matches = movies_clean["movieId"].isin(rating_summary["movieId"]).sum()
credits_matches = movies_clean["tmdbId"].isin(credits["tmdbId"]).sum()
keywords_matches = movies_clean["tmdbId"].isin(keywords["tmdbId"]).sum()

print()
print("Total movies:", movies_clean.shape[0])
print("Movies matched in rating_summary (movieId):", ratings_matches)
print("Movies matched in credits (tmdbId):        ", credits_matches)
print("Movies matched in keywords (tmdbId):       ", keywords_matches)

movieId type in movies_clean:   int64
movieId type in rating_summary: int64
tmdbId type in movies_clean:    int64
tmdbId type in credits:         int64
tmdbId type in keywords:        int64

Total movies: 9082
Movies matched in rating_summary (movieId): 9025
Movies matched in credits (tmdbId):         9082
Movies matched in keywords (tmdbId):        9082


Here I made sure 'movieId' and 'tmdbId' are stored as whole numbers in every table, so values line up correctly when the tables are joined.


In [24]:
print("ROW COUNTS BEFORE JOINING:")
print("ratings (one row per user rating): ", ratings.shape[0])
print("rating_summary (one row per movie):", rating_summary.shape[0])
print("movies_clean (one row per movie):  ", movies_clean.shape[0])
print("credits (one row per movie):       ", credits.shape[0])
print("keywords (one row per movie):      ", keywords.shape[0])

check_join_1 = movies_clean.merge(rating_summary, on="movieId", how="left")
check_join_2 = check_join_1.merge(credits, on="tmdbId", how="left")
check_join_3 = check_join_2.merge(keywords, on="tmdbId", how="left")

print()
print("ROW COUNTS AFTER EACH JOIN:")
print("After joining rating_summary:", check_join_1.shape[0])
print("After joining credits:       ", check_join_2.shape[0])
print("After joining keywords:      ", check_join_3.shape[0])

rated_not_in_movies = (rating_summary["movieId"].isin(movies_clean["movieId"]) == False).sum()
print("Rated movies not in the movies table:", rated_not_in_movies)

ROW COUNTS BEFORE JOINING:
ratings (one row per user rating):  99810
rating_summary (one row per movie): 9025
movies_clean (one row per movie):   9082
credits (one row per movie):        9082
keywords (one row per movie):       9082

ROW COUNTS AFTER EACH JOIN:
After joining rating_summary: 9082
After joining credits:        9082
After joining keywords:       9082
Rated movies not in the movies table: 0


Here I checked the row counts before and after each join. 'ratings' had 99,810 rows, one per user rating, and was aggregated into 'rating_summary'. 'movies_clean', 'credits', and 'keywords' each had 9,082 rows, one per movie. The row count stayed at 9,082 after joining rating_summary', then 'credits', then 'keywords'. 

In [25]:
duplicate_rating_keys = rating_summary["movieId"].duplicated().sum()

print("Rows in rating_summary:          ", rating_summary.shape[0])
print("Duplicate movieId in rating_summary:", duplicate_rating_keys)


Rows in rating_summary:           9025
Duplicate movieId in rating_summary: 0


- Duplicate key checks passed, so each row still represents exactly one movie.

### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [26]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check

,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782


In [27]:
genre_connection = sqlite3.connect(":memory:")
abt.to_sql("movies_for_sql", genre_connection, index=False)

genre_query = """
SELECT primary_genre,
       COUNT(*) AS number_of_movies,
       ROUND(AVG(user_rating_mean), 2) AS average_rating
FROM movies_for_sql
WHERE user_rating_count >= 20
GROUP BY primary_genre
ORDER BY average_rating DESC;
"""
genre_ratings = pd.read_sql_query(genre_query, genre_connection)
genre_connection.close()

print()
print("Average user rating by primary genre (movies with 20+ ratings):")
display(genre_ratings)


Average user rating by primary genre (movies with 20+ ratings):


,primary_genre,number_of_movies,average_rating
0,Documentary,5,4.00
1,Western,5,3.98
2,War,5,3.98
3,Mystery,16,3.90
4,Drama,293,3.75
5,Crime,82,3.73
6,History,6,3.59
7,Animation,38,3.59
8,Unknown,1,3.57
9,Family,12,3.54


- Updated business question: Among movies with enough MovieLens ratings to be reliable (20 or more), do average user ratings differ by genre, original language, and release decade?
- Cleaning decisions completed so far: I cleaned budget and revenue, to remove any values that were 0 or under $1,000 which were most likely data entry errors. I also cleaned runtime to remove any values that had 0 because movies cannot be that long. Additionally, vote count was cleaned by changing movies with 0 votes to missing. Genres was cleaned by labeling movies with no genre as "Unknown" so they are kept in the data but marked. Release status was cleaned by sorting the movies that are not relased yet. I removed the columns that were not necessary for our analysis such as genres, production_companies, and production_countries. "Primary_genres" was kept instead of "genres". No rows were lost in the cleaning process. 
- Transformation decisions completed so far:  I confirmed that each column already had the correct data type, cleaned up text fields by removing extra spaces and lowercasing language codes. Created 'language_group' (top 7 languages plus "other"), 'is_english', 'release_decade', and 'has_budget' to support comparing ratings by language and release period.
- Tables or files merged or joined: The 'ratings' table was aggregated to one row per movie by 'movieId' before joining, creating 'rating_summary' with each movie's average rating, rating count, standard deviation, and first and last rating dates. The cleaned movies table ('movies_clean') was then left-joined to 'rating_summary', 'credits', and 'keywords', so every movie is kept even if it has no ratings, credits, or keywords; 'links' was not joined because 'movies' already contains both 'movieId' and 'tmdbId'.
- Keys used for integration: 'movieId' joins the movies table to the aggregated ratings summary, and 'tmdbId' joins the movies table to 'credits' and 'keywords'. Each key has the same data type in both tables, and every join used 'validate="one_to_one"', which stops the code with an error if a key is duplicated on either side.
- SQL work: I loaded the ABT into an in-memory SQLite database and ran two queries. The provided check confirmed 9,082 rows and 9,082 unique 'movieId' values, and my own query calculated the average user rating by 'primary_genre' for the 1,300 movies with 20 or more ratings: among genres with many movies, Drama rated highest (3.75) and Horror rated lowest (3.35).
- Preliminary ABT unit of analysis: Each row represents one movie, with 'movieId' as the key that defines a row (tmdbId is also unique). The ratings table was aggregated to one row per movie before joining, so the ABT keeps all 9,082 movies and rating comparisons will use the movies with 20 or more ratings, and it was trimmed to the 12 columns needed for the business question. Rating comparisons will use the 1,300 movies with 20 or more ratings.
- Row count checks before and after joins: The cleaned movies table had 9,082 rows before joining and still had 9,082 rows after joining 'rating_summary', 'credits', and 'keywords', which matches the starter validation cell. 
- Duplicate key checks: All duplicate key checks passed: the source tables had 0 duplicate keys in Milestone 1, 'rating_summary' had 0 duplicate 'movieId' values. The final ABT has 0 duplicate 'movieId' and 0 duplicate 'tmdbId' values, confirming one row per movie
- Missingness checks after integration: After the joins, the columns needed for the business question ('primary_genre', 'original_language', 'language_group', and 'release_decade') have no missing values.
- Remaining data issues: Only 1,300 of 9,082 movies have 20 or more ratings, so the target is missing for most of the catalog, and several genres (Documentary, Western, War, History, and Music) and most non-English languages have too few eligible movies to compare reliably. Other open issues are that 'primary_genre' keeps only one genre per movie, older decades are sparse, budget and revenue are about half missing, post-release fields like 'vote_average' must be excluded as predictors to avoid leakage, and the very long runtimes still need a final decision before modeling. 

### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [32]:
columns_to_keep = [
       "movieId", "tmdbId", "title",
       "primary_genre",
       "language_group", "original_language",
       "release_decade", "release_year",
       "user_rating_mean", "user_rating_count",
       "rating_target_eligible", "high_user_rating",
   ]

print("Columns before:", abt.shape[1])
abt = abt[columns_to_keep]
print("Columns after: ", abt.shape[1])

Columns before: 12
Columns after:  12


In [33]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,,,,
1,tmdbId,int64,,,,
2,title,str,,,,
3,primary_genre,str,,,,
4,language_group,str,,,,


In [ ]:
cols = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

data_dictionary.loc[data_dictionary["column_name"] == "movieId", cols] = [
    "Unique MovieLens ID for each movie", "movies_metadata", "None",
    "Primary key; 0 duplicates in the ABT"
]

data_dictionary.loc[data_dictionary["column_name"] == "tmdbId", cols] = [
    "Unique TMDB ID for each movie", "movies_metadata", "None",
    "Used to join credits and keywords; 0 duplicates in the ABT"
]

data_dictionary.loc[data_dictionary["column_name"] == "title", cols] = [
    "Movie title", "movies_metadata", "Extra spaces removed",
    "Some titles repeat, so use movieId to identify movies"
]

data_dictionary.loc[data_dictionary["column_name"] == "primary_genre", cols] = [
    "First listed genre for the movie", "movies_metadata", "Missing values filled with Unknown",
    "35 movies are Unknown; ignores a movie's other genres; some genres have very few eligible movies"
]

data_dictionary.loc[data_dictionary["column_name"] == "language_group", cols] = [
    "Top 7 languages, with all others grouped as other", "Derived from original_language",
    "7 most common languages kept; the rest labeled other",
    "none identified"
]

data_dictionary.loc[data_dictionary["column_name"] == "original_language", cols] = [
    "Language code of the original film (for example, en, fr)", "movies_metadata",
    "Extra spaces removed and made lowercase",
    "cn and zh are both Chinese but were kept as separate labels"
]

data_dictionary.loc[data_dictionary["column_name"] == "release_decade", cols] = [
    "Decade the movie was released (for example, 1990)", "Derived from release_year",
    "(release_year // 10) * 10",
    "Early decades have very few movies with 20+ ratings"
]

data_dictionary.loc[data_dictionary["column_name"] == "release_year", cols] = [
    "Year the movie was released", "movies_metadata", "None",
    "Ranges from 1902 to 2016"
]

data_dictionary.loc[data_dictionary["column_name"] == "user_rating_mean", cols] = [
    "Average MovieLens user rating (0.5 to 5)", "ratings", "Mean of rating grouped by movieId",
    "Missing for 57 unrated movies; unreliable for movies with few ratings"
]

data_dictionary.loc[data_dictionary["column_name"] == "user_rating_count", cols] = [
    "Number of MovieLens ratings for the movie", "ratings", "Count of ratings grouped by movieId",
    "Missing (not 0) for the 57 unrated movies"
]

data_dictionary.loc[data_dictionary["column_name"] == "rating_target_eligible", cols] = [
    "True if the movie has 20 or more ratings", "Derived from user_rating_count",
    "user_rating_count >= 20",
    "1,300 of 9,082 movies are eligible"
]

data_dictionary.loc[data_dictionary["column_name"] == "high_user_rating", cols] = [
    "1 if the average rating is 3.75 or higher, 0 if lower",
    "Derived from user_rating_mean (starter join cell)",
    "user_rating_mean >= 3.75, only for eligible movies",
    "Missing for the 7,782 movies with fewer than 20 ratings; 3.75 is a chosen cutoff"
]



data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Unique MovieLens ID for each movie,movies_metadata,None,Primary key; 0 duplicates in the ABT
1,tmdbId,int64,Unique TMDB ID for each movie,movies_metadata,None,Used to join credits and keywords; 0 duplicate...
2,title,str,Movie title,movies_metadata,Extra spaces removed,"Some titles repeat, so use movieId to identify..."
3,primary_genre,str,First listed genre for the movie,movies_metadata,Missing values filled with Unknown,35 movies are Unknown; ignores a movie's other...
4,language_group,str,"Top 7 languages, with all others grouped as other",Derived from original_language,7 most common languages kept; the rest labeled...,none
5,original_language,str,Language code of the original film (for exampl...,movies_metadata,Extra spaces removed and made lowercase,cn and zh are both Chinese but were kept as se...
6,release_decade,int64,"Decade the movie was released (for example, 1990)",Derived from release_year,(release_year // 10) * 10,Early decades have very few movies with 20+ ra...
7,release_year,int64,Year the movie was released,movies_metadata,None,Ranges from 1902 to 2016
8,user_rating_mean,float64,Average MovieLens user rating (0.5 to 5),ratings,Mean of rating grouped by movieId,Missing for 57 unrated movies; unreliable for ...
9,user_rating_count,float64,Number of MovieLens ratings for the movie,ratings,Count of ratings grouped by movieId,Missing (not 0) for the 57 unrated movies


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [30]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,NaN,NaN,NaN,1992-06-23 04:19:23.796520,1902-09-01 00:00:00,1984-12-21 00:00:00,1997-09-01 00:00:00,2006-06-27 00:00:00,2016-09-23 00:00:00,NaN
runtime,9062.0,NaN,NaN,NaN,105.894615,2.0,93.0,102.0,115.0,1140.0,29.975022
budget,4599.0,NaN,NaN,NaN,32875873.121548,2500.0,6000000.0,19000000.0,42000000.0,380000000.0,40945823.694003
revenue,4713.0,NaN,NaN,NaN,94784392.707405,4631.0,9482579.0,34327391.0,104945305.0,2787965087.0,168336821.40531
popularity,9082.0,NaN,NaN,NaN,7.268905,0.000004,2.954521,6.418871,9.861742,547.488298,9.372314


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
